In [ ]:
!pip install requests

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

# ============================================================
# PULL ECB YIELD CURVE DATA
# ============================================================
# ECB publishes AAA-rated euro area sovereign yield curves daily
# We pull directly from their public API

def get_ecb_yields():
    """
    Pull ECB AAA euro area yield curve data.
    Maturities: 3m, 6m, 1y, 2y, 3y, 5y, 7y, 10y, 20y, 30y
    """
    
    # ECB SDW API endpoint for yield curve data
    # Series: YC.B.U2.EUR.4F.G_N_A.SV_C_YM — AAA rated, spot rates
    maturities = {
        "3M":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3M",
        "6M":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_6M",
        "1Y":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_1Y",
        "2Y":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y",
        "3Y":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_3Y",
        "5Y":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y",
        "7Y":  "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_7Y",
        "10Y": "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y",
        "20Y": "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_20Y",
        "30Y": "YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_30Y",
    }
    
    all_data = {}
    
    for label, series_key in maturities.items():
        url = (f"https://data-api.ecb.europa.eu/service/data/"
               f"YC/{series_key.split('YC.')[1]}"
               f"?format=csvdata&startPeriod=2020-01-01")
        
        try:
            response = requests.get(url, timeout=15)
            if response.status_code == 200:
                from io import StringIO
                df = pd.read_csv(StringIO(response.text))
                # ECB CSV has TIME_PERIOD and OBS_VALUE columns
                df = df[["TIME_PERIOD", "OBS_VALUE"]].copy()
                df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"])
                df = df.set_index("TIME_PERIOD")
                all_data[label] = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
                print(f"  ✅ {label} loaded — {len(df)} observations")
            else:
                print(f"  ❌ {label} failed — status {response.status_code}")
        except Exception as e:
            print(f"  ❌ {label} error — {e}")
    
    return pd.DataFrame(all_data).dropna(how="all")

print("Pulling ECB yield curve data...")
print("=" * 45)
yields = get_ecb_yields()
print("=" * 45)
print(f"\nData loaded: {len(yields)} trading days")
print(f"From: {yields.index[0].date()} to {yields.index[-1].date()}")

# ============================================================
# PLOT CURRENT YIELD CURVE
# ============================================================

# Maturity in years for x axis
mat_years = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
mat_labels = ["3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

current = yields.iloc[-1]
one_year_ago = yields.iloc[-253] if len(yields) > 253 else yields.iloc[0]
two_years_ago = yields.iloc[-505] if len(yields) > 505 else yields.iloc[0]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=mat_years, y=two_years_ago.values,
    mode="lines+markers", name="2 years ago",
    line=dict(color="rgba(100,149,237,0.4)", width=1.5, dash="dot"),
    marker=dict(size=5)
))

fig.add_trace(go.Scatter(
    x=mat_years, y=one_year_ago.values,
    mode="lines+markers", name="1 year ago",
    line=dict(color="#f39c12", width=1.5, dash="dash"),
    marker=dict(size=5)
))

fig.add_trace(go.Scatter(
    x=mat_years, y=current.values,
    mode="lines+markers", name="Current",
    line=dict(color="#28a745", width=2.5),
    marker=dict(size=8),
    hovertemplate="Maturity: %{x}Y<br>Yield: %{y:.3f}%<extra></extra>"
))

fig.update_layout(
    title=dict(
        text="EUR AAA Sovereign Yield Curve — ECB Data",
        x=0.5, font=dict(color="white", size=15)
    ),
    xaxis=dict(
        title="Maturity (years)",
        tickvals=mat_years,
        ticktext=mat_labels,
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)"
    ),
    yaxis=dict(
        title="Yield (%)",
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)"
    ),
    template="plotly_dark",
    height=500,
    legend=dict(font=dict(color="white"))
)

fig.show()

print(f"\n  Current EUR Yield Curve ({yields.index[-1].date()})")
print(f"  {'='*40}")
for label, val in zip(mat_labels, current.values):
    print(f"  {label:<6} {val:.3f}%")

In [ ]:
from scipy.optimize import curve_fit
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# NELSON-SIEGEL MODEL
# ============================================================

def nelson_siegel(maturity, beta0, beta1, beta2, tau):
    """
    Nelson-Siegel yield curve model.
    
    beta0 : level — long run yield
    beta1 : slope — short end loading
    beta2 : curvature — medium term hump
    tau   : decay factor — where the hump peaks
    """
    factor1 = (1 - np.exp(-maturity / tau)) / (maturity / tau)
    factor2 = factor1 - np.exp(-maturity / tau)
    
    return beta0 + beta1 * factor1 + beta2 * factor2

def fit_nelson_siegel(maturities, yields):
    """
    Fit Nelson-Siegel to observed yields.
    Returns fitted parameters and fitted curve.
    """
    # Initial parameter guesses
    p0 = [3.0, -1.0, 1.0, 2.0]
    bounds = (
        [-10, -10, -10, 0.1],   # lower bounds
        [ 10,  10,  10, 30.0]   # upper bounds
    )
    
    try:
        params, _ = curve_fit(
            nelson_siegel, maturities, yields,
            p0=p0, bounds=bounds, maxfev=10000
        )
        fitted = nelson_siegel(np.array(maturities), *params)
        rmse = np.sqrt(np.mean((fitted - yields)**2))
        return params, fitted, rmse
    except Exception as e:
        return None, None, None

# ============================================================
# FIT TO CURRENT CURVE
# ============================================================

mat_years = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
current_yields = yields.iloc[-1].values.astype(float)

params, fitted_yields, rmse = fit_nelson_siegel(mat_years, current_yields)
beta0, beta1, beta2, tau = params

print("=" * 50)
print("  Nelson-Siegel Model — Current Fit")
print("=" * 50)
print(f"  β₀ (Level):      {beta0:.4f}%  — long run yield")
print(f"  β₁ (Slope):      {beta1:.4f}%  — curve steepness")
print(f"  β₂ (Curvature):  {beta2:.4f}%  — mid-curve hump")
print(f"  τ  (Decay):      {tau:.4f}    — hump location")
print(f"  RMSE:            {rmse:.6f}%  — fit quality")
print("=" * 50)

# ============================================================
# PLOT FITTED VS ACTUAL
# ============================================================

# Smooth fitted curve
mat_smooth = np.linspace(0.25, 30, 300)
fitted_smooth = nelson_siegel(mat_smooth, *params)

fig = go.Figure()

# Actual yields
fig.add_trace(go.Scatter(
    x=mat_years, y=current_yields,
    mode="markers", name="Actual yields",
    marker=dict(color="#28a745", size=10, symbol="circle"),
    hovertemplate="Maturity: %{x}Y<br>Actual: %{y:.3f}%<extra></extra>"
))

# Fitted curve
fig.add_trace(go.Scatter(
    x=mat_smooth, y=fitted_smooth,
    mode="lines", name="Nelson-Siegel fit",
    line=dict(color="#4a9edd", width=2.5),
    hovertemplate="Maturity: %{x:.1f}Y<br>Fitted: %{y:.3f}%<extra></extra>"
))

# Residuals as vertical lines
for m, actual, fitted in zip(mat_years, current_yields, fitted_yields):
    fig.add_shape(
        type="line",
        x0=m, x1=m,
        y0=actual, y1=fitted,
        line=dict(color="#dc3545", width=1.5, dash="dot")
    )

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    name="Residuals (mispricing)",
    line=dict(color="#dc3545", width=1.5, dash="dot")
))

fig.update_layout(
    title=dict(
        text=f"Nelson-Siegel Fit — EUR AAA Curve ({yields.index[-1].date()})",
        x=0.5, font=dict(color="white", size=15)
    ),
    xaxis=dict(
        title="Maturity (years)",
        tickvals=mat_years,
        ticktext=["3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y","30Y"],
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)"
    ),
    yaxis=dict(
        title="Yield (%)",
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)"
    ),
    template="plotly_dark",
    height=500,
    legend=dict(font=dict(color="white"))
)

fig.show()

# ============================================================
# RESIDUALS TABLE — WHERE IS THE CURVE MISPRICED?
# ============================================================

print(f"\n  Residuals — Actual vs Model (basis points)")
print(f"  {'='*45}")
print(f"  {'Maturity':<10} {'Actual':>10} {'Model':>10} {'Residual':>12}")
print(f"  {'-'*45}")

labels = ["3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y","30Y"]
for label, actual, fit in zip(labels, current_yields, fitted_yields):
    residual_bps = (actual - fit) * 100
    direction = "CHEAP" if residual_bps > 2 else "RICH" if residual_bps < -2 else "FAIR"
    print(f"  {label:<10} {actual:>10.3f}% {fit:>10.3f}% "
          f"{residual_bps:>8.1f}bps  {direction}")

In [ ]:
# ============================================================
# STEP 3 — ROLLING PARAMETER ESTIMATION
# ============================================================

print("Fitting Nelson-Siegel to every trading day...")
print("This will take 30-60 seconds...")
print("=" * 50)

results = []

for date, row in yields.iterrows():
    obs = row.values.astype(float)
    
    # Skip days with too many missing values
    if np.sum(np.isnan(obs)) > 3:
        continue
    
    # Use available maturities only
    valid = ~np.isnan(obs)
    mats  = np.array(mat_years)[valid]
    ylds  = obs[valid]
    
    params_i, fitted_i, rmse_i = fit_nelson_siegel(mats.tolist(), ylds)
    
    if params_i is not None:
        results.append({
            "date":      date,
            "beta0":     params_i[0],
            "beta1":     params_i[1],
            "beta2":     params_i[2],
            "tau":       params_i[3],
            "rmse":      rmse_i,
            **{f"actual_{l}": y for l, y in zip(labels, obs)},
            **{f"fitted_{l}": f for l, f in zip(labels, fitted_i) if fitted_i is not None}
        })

params_df = pd.DataFrame(results).set_index("date")

print(f"  ✅ Fitted {len(params_df)} trading days successfully")
print(f"  From: {params_df.index[0].date()} to {params_df.index[-1].date()}")

# ============================================================
# PLOT PARAMETERS OVER TIME
# ============================================================

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=(
        "β₀ — Level (Long Run Yield %)",
        "β₁ — Slope (Curve Steepness %)",
        "β₂ — Curvature (Mid-Curve Hump %)",
        "Fit Quality (RMSE in bps)"
    ),
    vertical_spacing=0.08,
    row_heights=[0.28, 0.28, 0.28, 0.16]
)

# --- Beta 0 — Level ---
fig.add_trace(go.Scatter(
    x=params_df.index, y=params_df["beta0"],
    mode="lines", name="β₀ Level",
    line=dict(color="#28a745", width=1.5),
    hovertemplate="%{x}<br>β₀: %{y:.3f}%<extra></extra>"
), row=1, col=1)

# --- Beta 1 — Slope ---
fig.add_trace(go.Scatter(
    x=params_df.index, y=params_df["beta1"],
    mode="lines", name="β₁ Slope",
    line=dict(color="#4a9edd", width=1.5),
    hovertemplate="%{x}<br>β₁: %{y:.3f}%<extra></extra>"
), row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="white",
              line_width=0.8, row=2, col=1)

# --- Beta 2 — Curvature ---
fig.add_trace(go.Scatter(
    x=params_df.index, y=params_df["beta2"],
    mode="lines", name="β₂ Curvature",
    line=dict(color="#f39c12", width=1.5),
    hovertemplate="%{x}<br>β₂: %{y:.3f}%<extra></extra>"
), row=3, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="white",
              line_width=0.8, row=3, col=1)

# --- RMSE ---
fig.add_trace(go.Scatter(
    x=params_df.index, y=params_df["rmse"] * 100,
    mode="lines", name="RMSE (bps)",
    line=dict(color="#dc3545", width=1.2),
    fill="tozeroy",
    fillcolor="rgba(220,53,69,0.15)",
    hovertemplate="%{x}<br>RMSE: %{y:.2f}bps<extra></extra>"
), row=4, col=1)

# --- ECB key dates ---
ecb_events = {
    "ECB peak rate": "2023-09-20",
    "First cut":     "2024-06-06",
}

for label_ecb, date_ecb in ecb_events.items():
    for row_n in [1, 2, 3]:
        fig.add_vline(
            x=date_ecb,
            line_dash="dot",
            line_color="rgba(255,255,255,0.3)",
            line_width=1,
            row=row_n, col=1
        )
    fig.add_annotation(
        x=date_ecb, y=1, yref="paper",
        text=label_ecb,
        showarrow=False,
        font=dict(color="rgba(255,255,255,0.6)", size=9),
        textangle=-90,
        xshift=10
    )

fig.update_layout(
    title=dict(
        text="Nelson-Siegel Parameters — EUR AAA Curve (2020–2026)",
        x=0.5, font=dict(color="white", size=15)
    ),
    template="plotly_dark",
    height=900,
    showlegend=True,
    legend=dict(font=dict(color="white")),
    hovermode="x unified"
)

fig.show()

# ============================================================
# PRINT PARAMETER SUMMARY
# ============================================================

print(f"\n  Parameter Summary")
print(f"  {'='*55}")
print(f"  {'Param':<12} {'Current':>10} {'1Y Ago':>10} {'2Y Ago':>10} {'Mean':>10}")
print(f"  {'-'*55}")

param_names = ["beta0", "beta1", "beta2"]
param_labels = ["β₀ Level", "β₁ Slope", "β₂ Curve"]

for pname, plabel in zip(param_names, param_labels):
    current_p  = params_df[pname].iloc[-1]
    y1_ago     = params_df[pname].iloc[-253] if len(params_df) > 253 else params_df[pname].iloc[0]
    y2_ago     = params_df[pname].iloc[-505] if len(params_df) > 505 else params_df[pname].iloc[0]
    mean_p     = params_df[pname].mean()
    print(f"  {plabel:<12} {current_p:>10.3f} {y1_ago:>10.3f} {y2_ago:>10.3f} {mean_p:>10.3f}")

print(f"\n  Current curve interpretation:")
b0 = params_df["beta0"].iloc[-1]
b1 = params_df["beta1"].iloc[-1]
b2 = params_df["beta2"].iloc[-1]

print(f"  β₀ = {b0:.3f}% — long run yield the curve is anchored to")
print(f"  β₁ = {b1:.3f}%  — {'steepening' if b1 < 0 else 'inverted'} curve")
print(f"  β₂ = {b2:.3f}% — {'positive hump' if b2 > 0 else 'negative hump'} in the belly")

In [ ]:
# ============================================================
# STEP 4 — ANIMATED CURVE EVOLUTION + RV SIGNAL DASHBOARD
# ============================================================

# --- Part A: Animated yield curve evolution ---

print("Building animated curve evolution...")

# Sample every 5 trading days for performance
sampled = params_df.iloc[::5]

frames = []
slider_steps = []

for i, (date, row) in enumerate(sampled.iterrows()):
    b0, b1, b2, tau_i = row["beta0"], row["beta1"], row["beta2"], row["tau"]
    
    # Fitted curve for this date
    fitted = nelson_siegel(mat_smooth, b0, b1, b2, tau_i)
    
    # Actual yields for this date
    actual_yields = [row.get(f"actual_{l}", np.nan) for l in labels]
    
    frame = go.Frame(
        data=[
            go.Scatter(
                x=mat_smooth, y=fitted,
                mode="lines",
                line=dict(color="#4a9edd", width=2.5),
                name="NS fitted"
            ),
            go.Scatter(
                x=mat_years, y=actual_yields,
                mode="markers",
                marker=dict(color="#28a745", size=8),
                name="Actual"
            )
        ],
        name=str(date.date()),
        layout=go.Layout(
            title=dict(
                text=f"EUR AAA Yield Curve — {date.strftime('%b %d %Y')}  |  "
                     f"β₀={b0:.2f}%  β₁={b1:.2f}%  β₂={b2:.2f}%"
            )
        )
    )
    frames.append(frame)
    
    slider_steps.append(dict(
        args=[[str(date.date())],
              {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
        label=str(date.year) if date.month == 1 else "",
        method="animate"
    ))

# Initial frame
init_row = sampled.iloc[0]
init_fitted = nelson_siegel(
    mat_smooth,
    init_row["beta0"], init_row["beta1"],
    init_row["beta2"], init_row["tau"]
)
init_actual = [init_row.get(f"actual_{l}", np.nan) for l in labels]

fig_anim = go.Figure(
    data=[
        go.Scatter(x=mat_smooth, y=init_fitted,
                   mode="lines", line=dict(color="#4a9edd", width=2.5),
                   name="NS fitted"),
        go.Scatter(x=mat_years, y=init_actual,
                   mode="markers", marker=dict(color="#28a745", size=8),
                   name="Actual")
    ],
    frames=frames
)

fig_anim.update_layout(
    title=dict(
        text=f"EUR AAA Yield Curve Evolution — {sampled.index[0].strftime('%b %Y')} to {sampled.index[-1].strftime('%b %Y')}",
        x=0.5, font=dict(color="white", size=14)
    ),
    xaxis=dict(
        title="Maturity (years)",
        tickvals=mat_years,
        ticktext=["3M","6M","1Y","2Y","3Y","5Y","7Y","10Y","20Y","30Y"],
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)",
        range=[0, 31]
    ),
    yaxis=dict(
        title="Yield (%)",
        tickfont=dict(color="white"),
        gridcolor="rgba(255,255,255,0.1)",
        range=[-1, 5]
    ),
    template="plotly_dark",
    height=550,
    legend=dict(font=dict(color="white")),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=1.15, x=0.5,
        xanchor="center",
        buttons=[
            dict(
                label="▶ Play",
                method="animate",
                args=[None, {"frame": {"duration": 80, "redraw": True},
                             "fromcurrent": True,
                             "transition": {"duration": 0}}]
            ),
            dict(
                label="⏸ Pause",
                method="animate",
                args=[[None], {"frame": {"duration": 0, "redraw": False},
                               "mode": "immediate"}]
            )
        ]
    )],
    sliders=[dict(
        active=0,
        steps=slider_steps,
        currentvalue=dict(prefix="Date: ", font=dict(color="white")),
        pad=dict(t=50)
    )]
)

fig_anim.show()
print("✅ Animation rendered!")

# ============================================================
# Part B: Relative Value Signal Dashboard
# ============================================================

print("\nBuilding RV signal dashboard...")

# Calculate residuals for every day
rv_data = {}

for label_m, mat_m in zip(labels, mat_years):
    actual_col  = f"actual_{label_m}"
    fitted_col  = f"fitted_{label_m}"
    if actual_col in params_df.columns and fitted_col in params_df.columns:
        rv_data[label_m] = (params_df[actual_col] - params_df[fitted_col]) * 100

rv_df = pd.DataFrame(rv_data)

# --- Plot RV signals heatmap ---
fig_rv = go.Figure()

fig_rv.add_trace(go.Heatmap(
    z=rv_df.values.T,
    x=rv_df.index,
    y=labels,
    colorscale=[
        [0.0,  "rgb(180, 30, 30)"],    # very rich — dark red
        [0.35, "rgb(220, 80, 30)"],    # rich — orange
        [0.5,  "rgb(20,  20, 40)"],    # fair — dark
        [0.65, "rgb(30, 100, 180)"],   # cheap — blue
        [1.0,  "rgb(20, 180, 100)"],   # very cheap — green
    ],
    zmid=0,
    zmin=-15,
    zmax=15,
    colorbar=dict(
        title="Residual (bps)",
        tickfont=dict(color="white")
    ),
    hovertemplate="%{x}<br>%{y}: %{z:.1f}bps<extra></extra>"
))

fig_rv.update_layout(
    title=dict(
        text="EUR Yield Curve Relative Value Heatmap — CHEAP(green) vs RICH(red)",
        x=0.5, font=dict(color="white", size=14)
    ),
    xaxis=dict(
        title="Date",
        tickfont=dict(color="white")
    ),
    yaxis=dict(
        title="Maturity",
        tickfont=dict(color="white")
    ),
    template="plotly_dark",
    height=450
)

fig_rv.show()

# --- Current RV signals ---
print(f"\n  Current Relative Value Signals ({yields.index[-1].date()})")
print(f"  {'='*50}")
print(f"  {'Maturity':<10} {'Residual':>12} {'Signal':>10} {'Trade':>25}")
print(f"  {'-'*50}")

trade_map = {
    "3M":  "receive 3M OIS",
    "6M":  "receive 6M OIS",
    "1Y":  "receive 1Y swap",
    "2Y":  "receive 2Y swap",
    "3Y":  "receive 3Y swap",
    "5Y":  "pay 5Y swap",
    "7Y":  "pay 7Y swap",
    "10Y": "pay 10Y swap",
    "20Y": "receive 20Y swap",
    "30Y": "pay 30Y swap",
}

for label_m in labels:
    if label_m in rv_df.columns:
        res = rv_df[label_m].iloc[-1]
        if res > 5:
            signal = "CHEAP 🟢"
            trade  = f"Long / {trade_map[label_m]}"
        elif res < -5:
            signal = "RICH  🔴"
            trade  = f"Short / {trade_map.get(label_m, '')}"
        else:
            signal = "FAIR  ⚪"
            trade  = "No trade"
        print(f"  {label_m:<10} {res:>+10.1f}bps {signal:>10}  {trade}")

# --- Rolling z-score of residuals ---
print(f"\n  Building rolling z-score of residuals...")

fig_z = make_subplots(
    rows=2, cols=1,
    subplot_titles=(
        "2Y Residual Z-Score (rolling 60d) — steepener signal",
        "10Y Residual Z-Score (rolling 60d) — duration signal"
    ),
    vertical_spacing=0.15
)

for row_n, mat_label, color in [(1, "2Y", "#4a9edd"), (2, "10Y", "#f39c12")]:
    if mat_label in rv_df.columns:
        roll_mean = rv_df[mat_label].rolling(60).mean()
        roll_std  = rv_df[mat_label].rolling(60).std()
        zscore_rv = (rv_df[mat_label] - roll_mean) / roll_std

        fig_z.add_trace(go.Scatter(
            x=rv_df.index, y=zscore_rv,
            mode="lines", name=f"{mat_label} RV Z-score",
            line=dict(color=color, width=1.5),
            hovertemplate="%{x}<br>Z-score: %{y:.2f}<extra></extra>"
        ), row=row_n, col=1)

        for level, lcolor in [(2, "#dc3545"), (-2, "#dc3545"),
                               (1, "#fd7e14"), (-1, "#fd7e14"), (0, "white")]:
            fig_z.add_hline(
                y=level,
                line_dash="dash" if level == 0 else "dot",
                line_color=lcolor,
                line_width=1,
                row=row_n, col=1
            )

        # Shade positions
        fig_z.add_trace(go.Scatter(
            x=rv_df.index,
            y=zscore_rv.clip(lower=0),
            mode="lines", line=dict(width=0),
            fill="tozeroy",
            fillcolor="rgba(40,167,69,0.15)",
            showlegend=False, hoverinfo="skip"
        ), row=row_n, col=1)

        fig_z.add_trace(go.Scatter(
            x=rv_df.index,
            y=zscore_rv.clip(upper=0),
            mode="lines", line=dict(width=0),
            fill="tozeroy",
            fillcolor="rgba(220,53,69,0.15)",
            showlegend=False, hoverinfo="skip"
        ), row=row_n, col=1)

fig_z.update_layout(
    title=dict(
        text="EUR Yield Curve — Relative Value Z-Scores",
        x=0.5, font=dict(color="white", size=14)
    ),
    template="plotly_dark",
    height=600,
    hovermode="x unified",
    legend=dict(font=dict(color="white"))
)

fig_z.show()